# 07 · Reranker cross-encoder + Verificación FEVER-style

Demuestra las dos mejoras inspiradas en literatura científica:

- **Reranker cross-encoder** (Nogueira & Cho, 2019). BM25 propone 100 candidatos, un cross-encoder reordena el top-K.
- **Verificación FEVER-style** (Thorne et al., 2018). Para cada doc recuperado, NLI clasifica la relación claim↔evidencia en `SUPPORTS / REFUTES / NEI`.

Esto justifica el nombre VERITAS — *Verification and Retrieval Intelligent Truth Analysis System*.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src import config
from src.retrieval import BM25Retriever, CrossEncoderReranker, RerankedRetriever
from src.classification import EvidenceVerifier

df = pd.read_parquet(config.CORPUS_PATH).head(3000).reset_index(drop=True)
print('Corpus:', len(df))

## Etapa 1: BM25 solo (baseline)

In [ ]:
bm25 = BM25Retriever().fit(df['clean_text'].tolist())
query = 'covid vaccine causes infertility'
idx, scores = bm25.search(query, top_k=5)
print('=== BM25 ===')
for r, (i, s) in enumerate(zip(idx, scores), 1):
    print(f' [{r}] s={s:.2f}  {df.iloc[i]["title"][:90]}')

## Etapa 2: BM25 + Cross-Encoder Reranker

Patrón two-stage. El reranker descarga `cross-encoder/ms-marco-MiniLM-L-6-v2` (~22M params) la primera vez (~80MB).

In [ ]:
reranker = CrossEncoderReranker()
reranked = RerankedRetriever(
    base=bm25,
    reranker=reranker,
    candidate_pool=50,
    documents=df['text'].tolist(),
)
idx, scores = reranked.search(query, top_k=5)
print('=== BM25 + Reranker ===')
for r, (i, s) in enumerate(zip(idx, scores), 1):
    print(f' [{r}] s={s:.2f}  {df.iloc[i]["title"][:90]}')

## Etapa 3: Verificación FEVER-style con NLI

Para cada documento del top-3, clasificamos la relación claim ↔ evidencia.

In [ ]:
verifier = EvidenceVerifier()

top_evidences = df.iloc[idx[:3]]['text'].tolist()
for r, ev in enumerate(top_evidences, 1):
    result = verifier.verify(query, [ev])
    print(f'[{r}] {result.verdict}  scores={ {k: round(v,2) for k,v in result.scores.items()} }')
    print(f'    {ev[:200]}…\n')

## Evaluación cuantitativa: BM25 vs BM25+Reranker

Reportamos `P@5`, `NDCG@5`, `MAP` y `MRR` sobre el mismo conjunto de consultas que el notebook 06.

In [ ]:
import numpy as np
from src.evaluation import evaluate_retriever

queries, ground_truth = [], []
for subj, group in df.groupby('subject'):
    if len(group) < 10:
        continue
    sample = group.sample(min(2, len(group)), random_state=config.SEED)
    for _, row in sample.iterrows():
        queries.append(row['title'])
        ground_truth.append(group.index.tolist())
print('Queries:', len(queries))

rows = [
    evaluate_retriever(bm25, queries, ground_truth),
    evaluate_retriever(reranked, queries, ground_truth),
]
pd.DataFrame(rows).set_index('name').round(4)

## Referencias

- Nogueira, R. & Cho, K. (2019). *Passage Re-ranking with BERT.* arXiv:1901.04085
- Thorne, J. et al. (2018). *FEVER: a large-scale dataset for fact extraction and verification.* NAACL.
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT.* EMNLP.